<img src="../../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../../images/Konstanz_Logo.svg" width="200" /> <img src="../../images/KIT_Logo.png" width="200" />

# Convolutional Neural Networks

[Notebook 03](03_Refactoring_With_torch_nn.ipynb) ended with a training loop that
knows nothing about the model it trains. This notebook takes advantage of that: we
replace the single-layer classifier with a **convolutional network** and reuse `fit`
untouched.

Convolution itself is covered in
[01_TensorFlow/03_Image_Processing_With_CNNs](../01_TensorFlow/03_Image_Processing_With_CNNs/03_Image_Processing_With_CNNs.ipynb);
here the interest is in how PyTorch expresses it.

---

## Contents

1. [Starting point](#setup)
2. [A CNN with nn.Module](#cnn)
3. [nn.Sequential](#sequential)
4. [Wrapping the DataLoader](#wrapping)
5. [Using an accelerator](#accelerator)
6. [Closing thoughts](#closing)
7. [Resources](#resources)
8. [References](#references)

<a id="setup"></a>
## 1. Starting point

The data, the loaders and the training loop from notebook 03, so that this notebook
runs on its own.

In [1]:
import gzip
import pickle

import torch

with gzip.open("mnist.pkl.gz", "rb") as file:
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(file, encoding="latin-1")

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)

n, c = x_train.shape
print("training images:", tuple(x_train.shape))

training images: (50000, 784)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

bs = 64
epochs = 2
loss_func = F.cross_entropy

train_ds = TensorDataset(x_train, y_train)
valid_ds = TensorDataset(x_valid, y_valid)


def get_data(train_ds, valid_ds, bs):
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True),
        DataLoader(valid_ds, batch_size=bs * 2),
    )


def loss_batch(model, loss_func, xb, yb, opt=None):
    loss = loss_func(model(xb), yb)
    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()
    return loss.item(), len(xb)


def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss_batch(model, loss_func, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses, nums = zip(
                *[loss_batch(model, loss_func, xb, yb) for xb, yb in valid_dl]
            )
        val_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)
        print(f"epoch {epoch}   validation loss {val_loss:.5f}")


train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
print("batches per epoch:", len(train_dl))

batches per epoch: 782


<a id="cnn"></a>
## 2. A CNN with nn.Module

Three convolutional layers, each followed by a ReLU, then an average pooling at the
end. `nn.Conv2d` is PyTorch's convolution layer, and `stride=2` halves the width and
height at every step — the same trick used in the autoencoder.

One thing has to happen first. Our images are stored flat, 784 numbers in a row, and
a 2D convolution needs them as squares. **`view`** does that reshaping; it is
PyTorch's version of NumPy's `reshape`.

In [3]:
class Mnist_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1)

    def forward(self, xb):
        xb = xb.view(-1, 1, 28, 28)          # flat row -> 28 x 28 image
        xb = F.relu(self.conv1(xb))
        xb = F.relu(self.conv2(xb))
        xb = F.relu(self.conv3(xb))
        xb = F.avg_pool2d(xb, 4)
        return xb.view(-1, xb.size(1))

**Momentum** is a refinement of stochastic gradient descent: each update keeps a
little of the previous one, which smooths the path downhill and usually trains
faster.

Now the payoff. A completely different model — and `fit` is called exactly as
before.

In [4]:
lr = 0.1

model = Mnist_CNN()
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

epoch 0   validation loss 0.32193
epoch 1   validation loss 0.26969


<a id="sequential"></a>
## 3. nn.Sequential

The `Mnist_CNN` class does nothing but apply its layers in order, and for that
**`nn.Sequential`** is enough — the same idea as Keras's `Sequential` in the
TensorFlow notebooks.

There is one snag: the `view` in `forward` is not a layer, so it cannot go in the
list. A tiny wrapper turns any function into one.

In [5]:
class Lambda(nn.Module):
    def __init__(self, func):
        super().__init__()
        self.func = func

    def forward(self, x):
        return self.func(x)


def preprocess(x):
    return x.view(-1, 1, 28, 28)

In [6]:
model = nn.Sequential(
    Lambda(preprocess),
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.AvgPool2d(4),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

epoch 0   validation loss 0.30852
epoch 1   validation loss 0.29542


<a id="wrapping"></a>
## 4. Wrapping the DataLoader

That model is compact, but it only works on MNIST, for two reasons:

- it assumes the input is a flat vector of 784 numbers
- it assumes the grid is 4 × 4 by the time it reaches the pooling layer

Both can go. The reshaping belongs with the *data*, not the model, so we move it into
a wrapper around the `DataLoader` — anything that yields batches will do, so long as
it reshapes them on the way out.

In [7]:
def preprocess(x, y):
    return x.view(-1, 1, 28, 28), y


class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))


train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

The second assumption goes away by replacing `nn.AvgPool2d` with
**`nn.AdaptiveAvgPool2d`**, which is told the size of the *output* it should produce
rather than the size of the input it will get. The model then accepts any
single-channel 2D image.

In [8]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

epoch 0   validation loss 0.32750
epoch 1   validation loss 0.26250


<a id="accelerator"></a>
## 5. Using an accelerator

PyTorch runs on the CPU unless told otherwise. If a GPU is available, moving the data
and the model onto it usually makes training substantially faster — the difference
measured in the CNN notebook of the TensorFlow chapter.

Checking what is available:

In [9]:
device = (torch.accelerator.current_accelerator().type
          if torch.accelerator.is_available() else "cpu")

print(f"Using {device} device")

Using cpu device


Two things then have to move. The **batches**, which is another job for the wrapper
we just wrote:

In [10]:
def preprocess(x, y):
    return x.view(-1, 1, 28, 28).to(device), y.to(device)


train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

And the **model**. If `device` came back as `cpu` this changes nothing, and the code
runs regardless — which is why it is written this way rather than hard-coded.

In [11]:
model.to(device)
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

epoch 0   validation loss 0.23521
epoch 1   validation loss 0.16661


<a id="closing"></a>
## 6. Closing thoughts

We now have a general data pipeline and training loop that can train many kinds of
model. Along the way each of `torch.nn`, `torch.optim`, `Dataset` and `DataLoader`
was introduced by first doing without it, so to summarise what they are:

- **`torch.nn`**
  - **`Module`** — creates a callable that behaves like a function but can also hold
    state, such as layer weights. It knows which `Parameter`s it contains, and can
    zero all their gradients or loop through them for updates
  - **`Parameter`** — a wrapper telling a `Module` that this tensor is a weight
    needing updates during backpropagation. Only tensors with `requires_grad` set are
    updated
  - **`functional`** — usually imported as `F`; contains activation functions, loss
    functions, and stateless versions of layers such as convolutional and linear ones
- **`torch.optim`** — optimizers such as `SGD`, which update the `Parameter`s during
  the backward step
- **`Dataset`** — an abstract interface for objects with `__len__` and `__getitem__`,
  including PyTorch's own classes such as `TensorDataset`
- **`DataLoader`** — takes any `Dataset` and produces an iterator returning batches

There is plenty left to add — data augmentation, hyperparameter tuning, monitoring,
transfer learning. The [fastai](https://docs.fast.ai/) library was built with the
same design approach shown here and is a natural next step.

<a id="resources"></a>
## 7. Resources

Introduction to Artificial Neural Networks:

- https://youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&si=K6NmU277knsiknd7

- https://www.mzes.uni-mannheim.de/socialsciencedatalab/article/ann/

- https://www.geeksforgeeks.org/artificial-neural-networks-and-its-applications/



Autoencoders:

- https://towardsdatascience.com/introduction-to-autoencoders-7a47cf4ef14b

- https://www.tensorflow.org/tutorials/generative/autoencoder

- https://www.datacamp.com/tutorial/introduction-to-autoencoders



Convolutional Neural Networks:

- https://saturncloud.io/blog/a-comprehensive-guide-to-convolutional-neural-networks-the-eli5-way/

- https://www.youtube.com/watch?v=KuXjwB4LzSA&t=363s

- https://www.youtube.com/watch?v=py5byOOHZM8


Materials & Tutorials:

- https://www.tensorflow.org/tutorials/

- https://ki-kurs.org/ Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz

- https://huggingface.co/ Platform for tools for the creation of applications with machine learning.


Mixed - to be sorted:

- https://stackoverflow.com/a/27134600

- https://alexlenail.me/NN-SVG/index.html (creating NN-structure graphics)

- https://pytorch.org/tutorials/beginner/data_loading_tutorial.html -> example walkthrough creating a custom `FacialLandmarkDataset` class as a subclass of `Dataset`.

- https://pytorch.org/docs/stable/_modules/torch/utils/data/dataset.html#TensorDataset

- https://www.fast.ai/2017/11/13/validation-sets/ -> on validation

- https://www.quora.com/Does-the-order-of-training-data-matter-when-training-neural-networks -> on data shuffling

<a id="references"></a>
# References

The content of this workshop is in parts based on and inspired by the following sources:

* Python Course of the AG Peter (Prof. Dr. Christine Peter, Kevin Savade, Dr. Oleksandra Kukharenko, Dr. Andrej Berg)
* Software Carpentry workshops (https://software-carpentry.org/lessons/)
* Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz (https://ki-kurs.org/)
* Real Python (https://realpython.com/)
* Intro to Autoencoders (https://www.tensorflow.org/tutorials/generative/autoencoder)
* Image classification of MNIST using TensorFlow (https://www.kaggle.com/code/viratkothari/image-classification-of-mnist-using-tensorflow)
* The bwHPC wiki (https://wiki.bwhpc.de/)